<a href="https://colab.research.google.com/github/WellingtonRoque/MineracaoDados/blob/main/aulas/Aula06_Web_Scraping_Industria_4_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌐 Aula 06 — Web Scraping aplicado à Indústria 4.0

## Coleta de informações técnicas para análise de motores elétricos

**Disciplina:** ISW-039 — Mineração de Dados  
**Curso:** Desenvolvimento de Software Multiplataforma (DSM)  
**Ambiente:** Google Colab  
**Linguagem:** Python  
**Bibliotecas:** Requests, BeautifulSoup e Pandas

### 🎯 Objetivos
- Compreender Web Scraping e seu uso como fonte de dados.
- Interpretar a estrutura básica de HTML.
- Extrair informações com BeautifulSoup.
- Transformar dados extraídos em DataFrame.
- Limpar e transformar dados.
- Integrar dados externos aos dados dos sensores.
- Relacionar Web Scraping ao processo ETL.
- Identificar possíveis fontes Web para o próprio projeto.

> **Projeto didático:** continuaremos utilizando o monitoramento de motores elétricos como exemplo. Nesta aula, os dados dos sensores serão complementados por informações técnicas obtidas de uma página Web fictícia de um fabricante.

# 🏭 1. Relembrando nosso projeto

Nas aulas anteriores trabalhamos com dados de sensores de motores elétricos:

- corrente;
- tensão;
- vibração;
- temperatura;
- RPM;
- potência;
- horas de operação;
- status.

Esses dados mostram **o comportamento observado do motor**.

Agora imagine que a indústria também precise de informações externas:

- corrente nominal;
- tensão nominal;
- RPM nominal;
- potência nominal;
- temperatura máxima;
- vibração máxima.

Essas informações podem estar em documentação ou páginas técnicas de fabricantes.

Assim, podemos enriquecer nossa análise:

```text
Dados dos sensores
        +
Dados técnicos externos
        ↓
Base enriquecida
        ↓
Análise de dados
```

# 🌐 2. O que é Web Scraping?

Web Scraping é o processo automatizado de coleta de informações disponíveis em páginas da Web.

Fluxo:

```text
Página Web
   ↓
HTML
   ↓
Python
   ↓
BeautifulSoup
   ↓
Extração
   ↓
DataFrame
   ↓
Limpeza
   ↓
Análise
```

Web Scraping pode ser utilizado como uma estratégia de **extração de dados dentro de um processo ETL**.

# ⚠️ 3. Boas práticas

Antes de coletar dados de uma página real:

- verifique se a informação é pública;
- consulte os termos de uso;
- observe regras do site e `robots.txt`;
- evite requisições excessivas;
- respeite direitos autorais;
- evite coletar dados pessoais sem necessidade;
- verifique se existe uma API oficial.

> Nesta aula utilizaremos uma página **fictícia criada no notebook**, para praticar a técnica sem depender de um site externo.

# 💻 4. Preparando o ambiente

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

print("Bibliotecas carregadas com sucesso.")

# 🏭 5. Criando o Portal Técnico de Motores

Vamos simular uma página Web de um fabricante contendo informações técnicas de três modelos de motores.

In [ ]:
html = '''
<html>
<head>
    <title>Portal Técnico de Motores Industriais</title>
</head>
<body>

<h1>Portal Técnico de Motores Industriais</h1>

<div class="motor">
    <h2>MX-100</h2>
    <p class="potencia">7.5 kW</p>
    <p class="tensao">380 V</p>
    <p class="corrente">15 A</p>
    <p class="rpm">1750 RPM</p>
    <p class="temperatura_maxima">80 °C</p>
    <p class="vibracao_maxima">5.0 mm/s</p>
</div>

<div class="motor">
    <h2>MX-200</h2>
    <p class="potencia">11 kW</p>
    <p class="tensao">380 V</p>
    <p class="corrente">21 A</p>
    <p class="rpm">1750 RPM</p>
    <p class="temperatura_maxima">85 °C</p>
    <p class="vibracao_maxima">5.0 mm/s</p>
</div>

<div class="motor">
    <h2>MX-300</h2>
    <p class="potencia">15 kW</p>
    <p class="tensao">380 V</p>
    <p class="corrente">28 A</p>
    <p class="rpm">1760 RPM</p>
    <p class="temperatura_maxima">90 °C</p>
    <p class="vibracao_maxima">5.5 mm/s</p>
</div>

</body>
</html>
'''

print(html[:1000])

# 🔎 6. Entendendo o HTML

Exemplo:

```html
<h2>MX-100</h2>
<p class="potencia">7.5 kW</p>
<p class="temperatura_maxima">80 °C</p>
```

- `h2` representa o modelo.
- `p` representa um parágrafo.
- `class` ajuda a identificar o tipo de informação.

In [ ]:
soup = BeautifulSoup(html, "html.parser")

print(soup.title.text)

# 🧭 7. Localizando os motores

Cada motor está dentro de:

```html
<div class="motor">
```

Podemos localizar todos com `find_all()`.

In [ ]:
motores = soup.find_all("div", class_="motor")

print("Quantidade de motores encontrados:", len(motores))

# 🔍 8. Extraindo um motor

Vamos começar pelo primeiro motor.

In [ ]:
motor = motores[0]

print(motor.get_text(" ", strip=True))

In [ ]:
modelo = motor.find("h2").get_text(strip=True)
potencia = motor.find("p", class_="potencia").get_text(strip=True)
tensao = motor.find("p", class_="tensao").get_text(strip=True)
corrente = motor.find("p", class_="corrente").get_text(strip=True)
rpm = motor.find("p", class_="rpm").get_text(strip=True)
temperatura_maxima = motor.find("p", class_="temperatura_maxima").get_text(strip=True)
vibracao_maxima = motor.find("p", class_="vibracao_maxima").get_text(strip=True)

print("Modelo:", modelo)
print("Potência:", potencia)
print("Tensão:", tensao)
print("Corrente:", corrente)
print("RPM:", rpm)
print("Temperatura máxima:", temperatura_maxima)
print("Vibração máxima:", vibracao_maxima)

# 🔁 9. Extraindo todos os motores

Agora vamos automatizar a coleta.

In [ ]:
registros = []

for motor in motores:
    registros.append({
        "modelo": motor.find("h2").get_text(strip=True),
        "potencia": motor.find("p", class_="potencia").get_text(strip=True),
        "tensao": motor.find("p", class_="tensao").get_text(strip=True),
        "corrente_nominal": motor.find("p", class_="corrente").get_text(strip=True),
        "rpm_nominal": motor.find("p", class_="rpm").get_text(strip=True),
        "temperatura_maxima": motor.find("p", class_="temperatura_maxima").get_text(strip=True),
        "vibracao_maxima": motor.find("p", class_="vibracao_maxima").get_text(strip=True)
    })

registros

# 📊 10. Transformando os dados em DataFrame

In [ ]:
df_tecnico = pd.DataFrame(registros)

df_tecnico

Os valores foram extraídos como texto. Para análise, precisamos remover as unidades e convertê-los para números.

In [ ]:
df_tecnico["potencia"] = df_tecnico["potencia"].str.replace(" kW", "", regex=False).astype(float)
df_tecnico["tensao"] = df_tecnico["tensao"].str.replace(" V", "", regex=False).astype(float)
df_tecnico["corrente_nominal"] = df_tecnico["corrente_nominal"].str.replace(" A", "", regex=False).astype(float)
df_tecnico["rpm_nominal"] = df_tecnico["rpm_nominal"].str.replace(" RPM", "", regex=False).astype(int)
df_tecnico["temperatura_maxima"] = df_tecnico["temperatura_maxima"].str.replace(" °C", "", regex=False).astype(float)
df_tecnico["vibracao_maxima"] = df_tecnico["vibracao_maxima"].str.replace(" mm/s", "", regex=False).astype(float)

df_tecnico

In [ ]:
df_tecnico.dtypes

# 📈 11. Analisando os dados técnicos

In [ ]:
print("Maior potência:", df_tecnico["potencia"].max(), "kW")
print("Maior corrente nominal:", df_tecnico["corrente_nominal"].max(), "A")
print("Maior temperatura máxima:", df_tecnico["temperatura_maxima"].max(), "°C")
print("Maior vibração máxima:", df_tecnico["vibracao_maxima"].max(), "mm/s")

# 🔗 12. Recuperando os dados dos sensores

Vamos utilizar a mesma ideia do projeto didático: medições realizadas pelos sensores dos motores.

A coluna `modelo` será a chave utilizada para integrar as duas fontes.

In [ ]:
dados_sensores = {
    "motor": ["M001","M001","M001","M001","M001",
              "M002","M002","M002","M002","M002"],
    "modelo": ["MX-100","MX-100","MX-100","MX-100","MX-100",
               "MX-200","MX-200","MX-200","MX-200","MX-200"],
    "data_hora": [
        "2026-01-01 08:00","2026-01-01 09:00","2026-01-01 10:00",
        "2026-01-01 11:00","2026-01-01 12:00",
        "2026-01-01 08:00","2026-01-01 09:00","2026-01-01 10:00",
        "2026-01-01 11:00","2026-01-01 12:00"],
    "corrente": [12.4,12.8,13.7,15.2,15.8,10.8,11.0,11.2,11.5,11.7],
    "tensao": [380,379,381,378,377,380,381,380,379,380],
    "vibracao": [1.8,2.1,3.4,5.8,6.2,1.2,1.3,1.4,1.5,1.6],
    "temperatura": [62.3,64.1,68.7,76.4,79.2,55.1,56.0,56.8,57.4,58.2],
    "rpm": [1750,1748,1742,1728,1719,1752,1750,1749,1748,1747],
    "potencia": [7.2,7.4,8.1,9.2,9.6,6.1,6.2,6.3,6.4,6.5],
    "horas_operacao": [1250,1251,1252,1253,1254,820,821,822,823,824],
    "status": ["Normal","Normal","Alerta","Falha","Falha",
               "Normal","Normal","Normal","Normal","Normal"]
}

df_sensores = pd.DataFrame(dados_sensores)
df_sensores

# 🔄 13. Integrando as duas fontes

Temos:

**Dados dos sensores**

```text
motor, modelo, temperatura, vibração,
corrente, tensão, rpm, status...
```

**Dados técnicos**

```text
modelo, potência, tensão,
corrente nominal, rpm nominal,
temperatura máxima, vibração máxima
```

Como as duas bases possuem `modelo`, podemos utilizar `pd.merge()`.

In [ ]:
df_completo = pd.merge(
    df_sensores,
    df_tecnico,
    on="modelo",
    how="left"
)

df_completo

# 🚨 14. Comparando medições com limites técnicos

Agora podemos criar indicadores.

Primeiro, verificaremos se a temperatura medida ultrapassou a temperatura máxima do modelo.

In [ ]:
df_completo["temperatura_acima_limite"] = (
    df_completo["temperatura"] >
    df_completo["temperatura_maxima"]
)

df_completo[
    ["motor","modelo","temperatura","temperatura_maxima",
     "temperatura_acima_limite","status"]
]

In [ ]:
df_completo["vibracao_acima_limite"] = (
    df_completo["vibracao"] >
    df_completo["vibracao_maxima"]
)

df_completo[
    ["motor","modelo","vibracao","vibracao_maxima",
     "vibracao_acima_limite","status"]
]

In [ ]:
df_completo["corrente_acima_nominal"] = (
    df_completo["corrente"] >
    df_completo["corrente_nominal"]
)

df_completo[
    ["motor","modelo","corrente","corrente_nominal",
     "corrente_acima_nominal","status"]
]

# 📊 15. Criando um indicador percentual

Podemos calcular quanto da temperatura máxima está sendo utilizada:

```text
temperatura medida
------------------ × 100
temperatura máxima
```

In [ ]:
df_completo["temperatura_percentual"] = (
    df_completo["temperatura"] /
    df_completo["temperatura_maxima"]
) * 100

df_completo[
    ["motor","temperatura","temperatura_maxima",
     "temperatura_percentual"]
]

# 🧠 16. Interpretando

Agora podemos comparar:

- **o que o sensor mediu**;
- **qual é o limite técnico**;
- **qual é o status do motor**.

No nosso exemplo, o motor M001 apresenta aumento de temperatura, vibração e corrente ao longo das medições.

A integração com os dados técnicos permite enriquecer essa análise.

> **Web Scraping não é o objetivo final. É uma estratégia para obter dados que podem ajudar a responder perguntas do projeto.**

# 📝 17. Exercícios

### Exercício 1 — HTML
Identifique no HTML onde estão o modelo, potência, temperatura máxima e vibração máxima.

### Exercício 2 — BeautifulSoup
Crie o objeto `BeautifulSoup` e mostre o título da página.

### Exercício 3 — Quantidade
Conte quantos motores existem na página.

### Exercício 4 — Extração
Extraia todos os modelos e armazene-os em uma lista.

### Exercício 5 — DataFrame
Crie um DataFrame com modelo, potência, tensão, corrente nominal, RPM nominal, temperatura máxima e vibração máxima.

### Exercício 6 — Limpeza
Converta as informações numéricas para os tipos adequados.

### Exercício 7 — Análise
Descubra qual modelo possui maior potência e maior corrente nominal.

### Exercício 8 — Integração
Utilize `pd.merge()` para integrar `df_sensores` e `df_tecnico` pela coluna `modelo`.

### Exercício 9 — Indicadores
Crie os indicadores de temperatura acima do limite, vibração acima do limite e corrente acima da nominal.

### Exercício 10 — Interpretação
Analise o comportamento do M001 e explique o que os dados indicam.

# 🚀 18. Desafio — Aplicando ao seu projeto

Agora pense no projeto que seu grupo está desenvolvendo.

Pergunte:

> **Existe alguma informação pública na Web que poderia complementar os dados do meu projeto?**

Preencha:

| Item | Resposta |
|---|---|
| Problema do projeto | ... |
| Existe informação na Web? | ... |
| Qual informação? | ... |
| Qual seria a fonte? | ... |
| API ou Web Scraping? | ... |
| Como essa informação poderia complementar o projeto? | ... |

> Você não precisa realizar a coleta agora. O objetivo é identificar uma possível fonte de dados externa.

# 📌 19. Checklist

- [ ] Sei explicar o que é Web Scraping.
- [ ] Entendo HTML básico.
- [ ] Sei utilizar BeautifulSoup.
- [ ] Sei utilizar `find()` e `find_all()`.
- [ ] Sei extrair informações de uma página.
- [ ] Sei transformar os dados em DataFrame.
- [ ] Sei limpar dados extraídos.
- [ ] Sei integrar duas fontes com `merge()`.
- [ ] Entendo Web Scraping como parte do ETL.
- [ ] Sei comparar dados medidos com referências técnicas.
- [ ] Sei identificar uma possível fonte Web para meu projeto.

# 🎯 Conclusão

Nesta aula ampliamos nosso projeto:

```text
SENSORES
   ↓
Temperatura
Corrente
Tensão
Vibração
RPM
   +
   ↓
WEB SCRAPING
   ↓
Dados técnicos
   ↓
MERGE
   ↓
Base enriquecida
   ↓
Análise
```

Os dados dos sensores mostram **o que está acontecendo com o motor**.

Os dados técnicos mostram **valores de referência do equipamento**.

A combinação dessas fontes permite criar análises mais completas.

## 🗄️ Próxima aula

Na Aula 7 vamos trabalhar com:

> **Banco de Dados, SQL e integração com Pandas.**

Assim, continuaremos ampliando as fontes de dados do nosso projeto de Indústria 4.0.